In [25]:
import re
import pandas as pd
from datetime import datetime
from typing import List, Tuple

class Account:
    ALLOWED_OPERATIONS = set()

    def __init__(self, owner: str, account_number: str, initial_balance: float = 0.0):
        self.owner = owner
        self.account_number = account_number
        self._balance = 0.0
        self._transaction_history: List[Tuple[float, str, datetime]] = []
        if initial_balance < 0:
            raise ValueError("Начальный баланс не может быть отрицательным")
        if initial_balance > 0:
            self.deposit(initial_balance)

    @property
    def owner(self) -> str:
        return self._owner

    @owner.setter
    def owner(self, value: str):
        if not isinstance(value, str):
            raise ValueError("Имя владельца должно быть строкой")
        if not re.fullmatch(r'^[A-Za-zА-Яа-яЁё]+\s+[A-Za-zА-Яа-яЁё]+$', value):
            raise ValueError("Имя владельца должно быть в формате 'Имя Фамилия' кириллицей или латиницей")
        if not all(part[0].isupper() for part in value.split()):
            raise ValueError("Имя и фамилия должны начинаться с заглавной буквы")
        self._owner = value

    @property
    def balance(self) -> float:
        return self._balance

    def deposit(self, amount: float):
        if amount < 0:
            raise ValueError("Сумма пополнения не может быть отрицательной")
        self._balance += amount
        self._transaction_history.append((amount, 'deposit', datetime.now()))

    def withdraw(self, amount: float):
        if amount < 0:
            raise ValueError("Сумма снятия не может быть отрицательной")
        if amount > self._balance:
            raise ValueError("Недостаточно средств на счёте")
        self._balance -= amount
        self._transaction_history.append((-amount, 'withdraw', datetime.now()))

    def get_last_large_transactions(self, n: int) -> List[Tuple[float, str, datetime]]:
        if n <= 0:
            return []
        sorted_by_amount = sorted(
            self._transaction_history,
            key=lambda x: abs(x[0]),
            reverse=True
        )
        return sorted_by_amount[:n]

    def clean_history(self, raw_records: List[dict]) -> List[Tuple[float, str, datetime]]:
        clean_transactions = []
        invalid_reasons = []

        DATE_FORMATS = [
            '%Y-%m-%d %H:%M:%S',
            '%Y-%m-%d %H:%M',
            '%d/%m/%Y %H:%M',
            '%d/%m/%Y %H:%M:%S',
            '%Y-%m-%d',
        ]

        def parse_date(date_str: str) -> datetime:
            for fmt in DATE_FORMATS:
                try:
                    return datetime.strptime(date_str.strip(), fmt)
                except ValueError:
                    continue
            raise ValueError(f"Не удалось распарсить дату: {date_str}")

        for i, record in enumerate(raw_records):
            try:
                missing = [key for key in ['date', 'operation', 'amount', 'status']
                          if key not in record or pd.isna(record[key])]
                if missing:
                    raise ValueError(f"Missing fields: {missing}")

                status = str(record['status']).strip().lower()
                if status not in {'valid', 'success', 'succes', 'ok'}:
                    raise ValueError(f"Invalid status: {record['status']}")

                amount = float(record['amount'])
                if amount <= 0:
                    raise ValueError(f"Amount <= 0: {amount}")

                op_raw = str(record['operation']).strip().lower()
                corrections = {'diposit': 'deposit', 'withdrow': 'withdraw', 'int': 'interest'}
                op = corrections.get(op_raw, op_raw)

                if op not in self.ALLOWED_OPERATIONS:
                    raise ValueError(f"Unknown operation: {op_raw} → {op}")

                dt = parse_date(str(record['date']))

                adjusted_amount = -amount if op == 'withdraw' else amount
                clean_transactions.append((adjusted_amount, op, dt))

            except Exception as e:
                invalid_reasons.append(f"Record {i}: {e}")

        if invalid_reasons:
            print(f"\nОтброшено {len(invalid_reasons)} транзакций:")
            for r in invalid_reasons[:10]:
                print("  ", r)

        return clean_transactions

    def load_history_from_file(self, filepath: str):
        if filepath.endswith('.csv'):
            df = pd.read_csv(filepath)
        elif filepath.endswith('.json'):
            df = pd.read_json(filepath)
        else:
            raise ValueError("Поддерживаются только .csv и .json файлы")

        account_type_expected = self.account_type
        filtered_df = df[
            (df['account_number'] == self.account_number) &
            (df['account_type'] == account_type_expected)
        ].copy()

        raw_records = filtered_df.to_dict(orient='records')

        clean_transactions = self.clean_history(raw_records)

        clean_transactions.sort(key=lambda x: x[2])

        self._balance = 0.0
        self._transaction_history = []

        for amount, op, dt in clean_transactions:
            self._balance += amount
            self._transaction_history.append((amount, op, dt))


class CheckingAccount(Account):
    account_type = "checking"
    ALLOWED_OPERATIONS = {'deposit', 'withdraw'}

    def __init__(self, owner: str, account_number: str, initial_balance: float = 0.0):
        super().__init__(owner, account_number, initial_balance)


class SavingsAccount(Account):
    account_type = "savings"
    ALLOWED_OPERATIONS = {'deposit', 'withdraw', 'interest'}

    def __init__(self, owner: str, account_number: str, initial_balance: float = 0.0):
        super().__init__(owner, account_number, initial_balance)

    def apply_interest(self, rate: float):
        if rate < 0:
            raise ValueError("Процентная ставка не может быть отрицательной")
        interest = self._balance * (rate / 100)
        self._balance += interest
        self._transaction_history.append((interest, 'interest', datetime.now()))

    def withdraw(self, amount: float):
        if amount < 0:
            raise ValueError("Сумма снятия не может быть отрицательной")
        if amount > self._balance:
            raise ValueError("Недостаточно средств на счёте")
        if amount > self._balance * 0.5:
            raise ValueError("Нельзя снять более 50% от текущего баланса")
        self._balance -= amount
        self._transaction_history.append((-amount, 'withdraw', datetime.now()))

In [27]:
sa = SavingsAccount("Иван Иванов", "ACC-100001")
sa.load_history_from_file("transactions_dirty.json")

ca = CheckingAccount("Иван Иванов", "ACC-100001")
ca.load_history_from_file("transactions_dirty.json")

print("—"*40)
print("SavingsAccount:")
print("\t", "Баланс:", sa.balance)
print("\t", "Крупные транзакции:", sa.get_last_large_transactions(3))

print("—"*40)
print("CheckingAccount:")
print("\t", "Баланс:", ca.balance)
print("\t", "Крупные транзакции:", ca.get_last_large_transactions(3))


Отброшено 6 транзакций:
   Record 5: Missing fields: ['amount']
   Record 8: Missing fields: ['operation']
   Record 10: Unknown operation: interest → interest
   Record 13: Не удалось распарсить дату: 2025-17-34 12:00:00
   Record 16: Unknown operation: interest → interest
   Record 17: Не удалось распарсить дату: 2023-16-40 12:00:00
————————————————————————————————————————
SavingsAccount:
	 Баланс: 0.0
	 Крупные транзакции: []
————————————————————————————————————————
CheckingAccount:
	 Баланс: 4856.0
	 Крупные транзакции: [(921.0, 'deposit', datetime.datetime(2025, 9, 27, 22, 17, 26)), (916.0, 'deposit', datetime.datetime(2025, 10, 19, 22, 17, 26)), (880.0, 'deposit', datetime.datetime(2025, 9, 29, 22, 17, 26))]
